# (b) & (c)

### 1: Import & Load & Concatenate

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats

# Load stock return data from Excel files
stock_data1 = pd.read_excel(r"D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_1.xlsx", dtype={'Stkcd': str})
stock_data2 = pd.read_excel(r"D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_2.xlsx", dtype={'Stkcd': str})
stock_data = pd.concat([stock_data1, stock_data2], ignore_index=True)
stock_data = stock_data[stock_data['Trdwnt'] < '2023-01'].reset_index(drop=True)

# Load risk-free rate data
rf_data = pd.read_excel(r"D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\weekly_risk_free_rate.xlsx", sheet_name='Sheet1')
rf_data['trading_date_yw'] = pd.to_datetime(rf_data['trading_date_yw'])
rf_data.set_index('trading_date_yw', inplace=True)
rf_data['Trdwnt'] = rf_data.index.to_period('W').strftime('%Y-%U')

# Merge datasets
merged_df = pd.merge(stock_data, rf_data[['Trdwnt', 'risk_free_return']], on='Trdwnt', how='left')
print(merged_df.head())

    Stkcd   Trdwnt    Wretnd  risk_free_return
0  000001  2017-01  0.003297          0.000571
1  000001  2017-02  0.003286          0.000738
2  000001  2017-03   0.00655          0.000543
3  000001  2017-04  0.011931          0.000774
4  000001  2017-05 -0.007503          0.000802


### 2: Calculate market returns

In [12]:
market_returns = merged_df.groupby('Trdwnt')['Wretnd'].mean().rename('Market_Return').reset_index()
merged_df = pd.merge(merged_df, market_returns, on='Trdwnt', how='left')

# Sort by week
merged_df = merged_df.sort_values('Trdwnt')

# Display basic info
print("First few rows of merged DataFrame:")
print(merged_df.head())

First few rows of merged DataFrame:
         Stkcd   Trdwnt    Wretnd  risk_free_return Market_Return
0       000001  2017-01  0.003297          0.000571      0.016611
13592   000062  2017-01 -0.029567          0.000571      0.016611
429664  300016  2017-01  -0.00379          0.000571      0.016611
818649  600555  2017-01    0.0064          0.000571      0.016611
429966  300017  2017-01  0.020705          0.000571      0.016611


# (d)

### 1: Define Time Periods

In [13]:
unique_weeks = merged_df['Trdwnt'].unique()
total_weeks = len(unique_weeks)
print(f"Total unique weeks in dataset: {total_weeks}")

# Define periods: first 100 weeks, next 100 weeks, and remaining weeks
period1 = unique_weeks[:100]
period2 = unique_weeks[100:200] if total_weeks > 200 else unique_weeks[100:total_weeks//2]
period3 = unique_weeks[200:] if total_weeks > 200 else unique_weeks[total_weeks//2:]

# Assign period labels
merged_df['Period'] = merged_df['Trdwnt'].apply(
    lambda x: 1 if x in period1 else (2 if x in period2 else (3 if x in period3 else None))
)
merged_df = merged_df.dropna(subset=['Period'])

print(merged_df.head())

Total unique weeks in dataset: 308
         Stkcd   Trdwnt    Wretnd  risk_free_return Market_Return  Period
0       000001  2017-01  0.003297          0.000571      0.016611       1
13592   000062  2017-01 -0.029567          0.000571      0.016611       1
429664  300016  2017-01  -0.00379          0.000571      0.016611       1
818649  600555  2017-01    0.0064          0.000571      0.016611       1
429966  300017  2017-01  0.020705          0.000571      0.016611       1


### 2: Calculate Individual Betas in Period 1

In [15]:
period1_df = merged_df[merged_df['Period'] == 1].copy()

# Ensure numeric columns
period1_df['Wretnd'] = pd.to_numeric(period1_df['Wretnd'], errors='coerce')
period1_df['Market_Return'] = pd.to_numeric(period1_df['Market_Return'], errors='coerce')

# Drop rows with NaN in critical columns after conversion
period1_df = period1_df.dropna(subset=['Wretnd', 'Market_Return'])

def compute_stock_beta(data):
    if len(data) < 10:  # Require at least 10 observations
        return np.nan
    try:
        X = sm.add_constant(data['Market_Return'], has_constant=True)
        y = data['Wretnd']
        # Verify data before regression
        if not (y.dtype.kind in 'biuf' and X.values.dtype.kind in 'biuf'):
            print(f"Non-numeric data detected in stock {data['Stkcd'].iloc[0]}")
            print(f"y: {np.asarray(y)}")
            print(f"X: {np.asarray(X)}")
            return np.nan
        model = sm.OLS(y, X, missing='drop').fit()
        return model.params['Market_Return']
    except Exception as e:
        print(f"Error in stock {data['Stkcd'].iloc[0]}: {e}")
        return np.nan

# Compute betas for each stock
beta_results = period1_df.groupby('Stkcd').apply(compute_stock_beta)
beta_df = pd.DataFrame({'Beta_Period1': beta_results}).dropna()

# Merge betas back to main dataframe
merged_df = merged_df.merge(beta_df, on='Stkcd', how='left')

print(merged_df.head())

Error in stock 000001: has_constant must be a string
Error in stock 000002: has_constant must be a string
Error in stock 000004: has_constant must be a string
Error in stock 000005: has_constant must be a string
Error in stock 000006: has_constant must be a string
Error in stock 000007: has_constant must be a string
Error in stock 000008: has_constant must be a string
Error in stock 000009: has_constant must be a string
Error in stock 000010: has_constant must be a string
Error in stock 000011: has_constant must be a string
Error in stock 000012: has_constant must be a string
Error in stock 000014: has_constant must be a string
Error in stock 000016: has_constant must be a string
Error in stock 000017: has_constant must be a string
Error in stock 000018: has_constant must be a string
Error in stock 000019: has_constant must be a string
Error in stock 000020: has_constant must be a string
Error in stock 000021: has_constant must be a string
Error in stock 000022: has_constant must be a 

C:\Users\19927\AppData\Local\Temp\ipykernel_6120\564067309.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  beta_results = period1_df.groupby('Stkcd').apply(compute_stock_beta)


### 3: Form Portfolios

In [16]:
valid_stocks = merged_df.dropna(subset=['Beta_Period1'])
valid_stocks['Portfolio'] = pd.qcut(valid_stocks['Beta_Period1'], q=10, labels=False) + 1
portfolio_map = valid_stocks[['Stkcd', 'Portfolio']].drop_duplicates()
merged_df = merged_df.merge(portfolio_map, on='Stkcd', how='left')
merged_df = merged_df.dropna(subset=['Portfolio'])
merged_df['Portfolio'] = merged_df['Portfolio'].astype(int)

ValueError: Bin edges must be unique: Index([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan], dtype='float64', name='Beta_Period1').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [4]:
import statsmodels.api as sm

# Load and concatenate stock return data
stock_df1 = pd.read_excel("D:/MG/！CUHKSZ/~！大二 下/FIN3080/Assignments/A3/TRD_Week_1.xlsx")
stock_df2 = pd.read_excel("D:/MG/！CUHKSZ/~！大二 下/FIN3080/Assignments/A3/TRD_Week_2.xlsx")
stock_df = pd.concat([stock_df1, stock_df2], ignore_index=True)

# Calculate market return as the mean of all stocks' returns per week
market_return = stock_df.groupby('Trading Week')['Weekly Return Without Cash Dividend Reinvested'].mean().reset_index()
market_return.columns = ['Trading Week', 'Market Return']

KeyError: 'Trading Week'